# Paradox World Model (PWM) — train and benchmark against Google TimesFM

**Runtime → Change runtime type → T4 GPU** (free) or A100, then run the cells top to bottom.

What this notebook does:
1. Trains **PWM-Dyn** from scratch (random initialisation — no pretrained weights) on real data:
   California road sensors (PEMS08, PEMS04) and **IMD gridded daily rainfall over India**.
2. Runs **Google TimesFM 2.5 and 3.0** zero-shot on exactly the same test windows, in a *separate*
   environment, as opponents only.
3. Proves provenance: PWM's saved weights contain only PWM's own parameters.
4. Uses **gpt-oss** (via Groq) to explain PWM's forecasts and what-if scenarios.
5. Writes `RESULTS.md` with the full comparison.

## 1. Setup
Upload `pwm_kit.zip` with the folder icon on the left (or put it in Google Drive), then run:

In [ ]:
import os, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
ZIP = '/content/pwm_kit.zip'   # or '/content/drive/MyDrive/pwm_kit.zip'
USE_DRIVE = True               # keep models/results in Google Drive so a disconnect loses nothing
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    if not os.path.exists(ZIP) and os.path.exists('/content/drive/MyDrive/pwm_kit.zip'):
        ZIP = '/content/drive/MyDrive/pwm_kit.zip'
!unzip -q -o {ZIP} -d /content
%cd /content/pwm_kit
if USE_DRIVE:
    os.makedirs('/content/drive/MyDrive/PWM/models', exist_ok=True); os.makedirs('/content/drive/MyDrive/PWM/results', exist_ok=True)
    !rm -rf models results && ln -s /content/drive/MyDrive/PWM/models models && ln -s /content/drive/MyDrive/PWM/results results
!ls

## 2. Install
PWM needs only PyTorch + `imdlib` (data). TimesFM goes into a **separate** environment so PWM can never import it.

In [ ]:
!pip install -q imdlib virtualenv
!virtualenv -q --system-site-packages /content/opponents_env
!/content/opponents_env/bin/pip install -q "timesfm[torch]"
!python -c "import timesfm" 2>&1 | tail -1      # expected: ModuleNotFoundError (PWM side cannot see TimesFM)
!/content/opponents_env/bin/python -c "import timesfm, timesfm3; print('opponents env OK')"

## 3. Real data

In [ ]:
!mkdir -p data/raw
for d in ['PEMS08', 'PEMS04']:
    for ext in ['npz', 'csv']:
        !curl -sL -o data/raw/{d}.{ext} https://raw.githubusercontent.com/guoshnBJTU/ASTGNN/main/data/{d}/{d}.{ext}
REGION, Y0, Y1 = 'central_india', 2000, 2024      # also: 'north_india', 'west_coast'
from pwm.data import download_imd, load
download_imd(REGION, Y0, Y1)                      # IMD 0.25 deg daily rainfall (~25 MB per year)
for n in ['PEMS08', 'PEMS04', f'IMD_{REGION}']:
    print(load(n).describe())

## 4. Train PWM-Dyn (3 seeds per dataset)
Roughly 1–2 hours on a T4. Each run saves to `models/`. Validation decides promotion; test is never used for selection.

In [ ]:
SEEDS = [0, 1, 2]
DATASETS = ['PEMS08', 'PEMS04', f'IMD_{REGION}']
EPOCHS = {'PEMS08': 60, 'PEMS04': 60, f'IMD_{REGION}': 80}
for ds in DATASETS:
    for s in SEEDS:
        if os.path.exists(f'models/{ds}/v0.3-s{s}/weights.pt'):
            print('skip', ds, s); continue          # resume after a disconnect
        !python train.py --data {ds} --version v0.3-s{s} --seed {s} --epochs {EPOCHS[ds]} 2>&1 | tail -3

## 5. Score PWM (seed ensemble) on the held-out test period

In [ ]:
STRIDE = 1      # use 3 to save time — PWM and TimesFM MUST use the same stride
for ds in DATASETS:
    !python evaluate_pwm.py --data {ds} --stride {STRIDE}

## 6. Opponents: Google TimesFM, zero-shot
TimesFM gets a **much longer history** than PWM (512 steps ≈ 43 h of traffic; 365 days of rain), which favours TimesFM.
TimesFM 2.5 weights are Apache-2.0; TimesFM 3.0 weights are **non-commercial research only** — fine for a benchmark, never ship them.

In [ ]:
OPP = '/content/opponents_env/bin/python opponents/run_timesfm.py'
CTX = {'PEMS08': 512, 'PEMS04': 512, f'IMD_{REGION}': 365}
for ds in DATASETS:
    !{OPP} --data {ds} --model tfm25 --ctx {CTX[ds]} --stride {STRIDE}
    !{OPP} --data {ds} --model tfm3  --ctx {CTX[ds]} --stride {STRIDE}
    !{OPP} --data {ds} --model tfm3  --ctx {CTX[ds]} --stride {STRIDE} --mode multivariate

## 7. Provenance check — must say PASS

In [ ]:
!python check_provenance.py | tail -3

## 8. Results

In [ ]:
!python compare.py > /dev/null
from IPython.display import Markdown, display
display(Markdown(open('RESULTS.md').read()))

## 9. gpt-oss reasoning layer
Add your Groq key in Colab **Secrets** (key icon on the left) as `GROQ_API_KEY`. gpt-oss only *explains* PWM's numbers.
What-if outputs are the model's simulated response and are **not validated** against real interventions.

In [ ]:
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
!python reason/gptoss_layer.py explain --data PEMS08 --window 500 --node 12 --reveal_truth | tail -25
!python reason/gptoss_layer.py whatif --data PEMS08 --window 500 --nodes 12 13 --scenario closure | tail -25
!python reason/gptoss_layer.py whatif --data IMD_{REGION} --window 200 --nodes 10 11 12 --scenario scale --factor 1.5 | tail -25

## 10. Download everything

In [ ]:
!zip -qr /content/pwm_results.zip RESULTS.md PROVENANCE_REPORT.json results models/PROMOTED.json models/*/v0.3-*/report.json
from google.colab import files; files.download('/content/pwm_results.zip')